# GRACE universal potentials in LAMMPS, via `pylammpsmpi`

This notebook drives the same `pair_style grace/fs` potential as [`01_lammps_grace_with_lammpsparser.ipynb`](01_lammps_grace_with_lammpsparser.ipynb), but through [`pylammpsmpi`](https://pylammpsmpi.readthedocs.io/en/latest/) instead of writing a LAMMPS input file.

`pylammpsmpi` controls a `mpi4py`-parallel LAMMPS instance from an ordinary (serial) Python process or notebook. Its `LammpsLibrary` interface exposes LAMMPS input-script commands as Python method/attribute calls (`lmp.pair_style(...)`, `lmp.run(...)`, ...) and thermodynamic keywords as properties (`lmp.pe`, `lmp.temp`, ...), which is convenient for anyone already familiar with LAMMPS input files who wants to drive a calculation interactively or step-by-step instead of through a static script.

We use `grace/fs` - the native, TensorFlow-free GRACE evaluator - because the `conda-forge` LAMMPS build does not link `libtensorflow` and therefore does not compile the TensorFlow-based `pair_style grace`.

## 1. Get a GRACE/FS foundation model

`grace/fs` reads a native YAML file exported from a model *checkpoint* with `grace_utils` (see the other notebook for why). Skip this cell if you already ran it there - the `~/.cache/grace` cache is shared.

In [ ]:
import os

model_name = "GRACE-FS-OAM"
checkpoint_dir = os.path.expanduser(os.path.join("~/.cache/grace/checkpoints", model_name))
fs_model_path = os.path.join(checkpoint_dir, "FS_model.yaml")

!grace_models checkpoint {model_name}
!grace_utils -p {checkpoint_dir}/model.yaml -c {checkpoint_dir}/checkpoint export -sf -n {fs_model_path}

## 2. Start a LAMMPS instance

`LammpsLibrary(cores=1)` launches an `mpi4py` LAMMPS process (via `openmpi`/`mpiexec`) in the background and gives us a handle to it.

In [ ]:
from pylammpsmpi import LammpsLibrary

element = "Al"
lmp = LammpsLibrary(cores=1)

## 3. Build an fcc aluminium cell and attach the GRACE/FS potential

Instead of loading a structure from a data file, we build it directly with LAMMPS' own `lattice`/`region`/`create_atoms` commands - the classic LAMMPS-input-script way of setting up a calculation.

Note: `create_atoms` is also the name of a lower-level array-based method on `LammpsLibrary` (mirroring the LAMMPS C library API), so to send it as a plain input-script command we go through `lmp.command(...)` explicitly; every other command below is available directly as an attribute.

In [ ]:
lmp.units("metal")
lmp.atom_style("atomic")
lmp.boundary("p", "p", "p")

a0 = 4.05  # Angstrom, approximate fcc Al lattice constant
lmp.lattice("fcc", a0)
lmp.region("box", "block", 0, 3, 0, 3, 0, 3)
lmp.create_box(1, "box")
lmp.command("create_atoms 1 box")
lmp.mass(1, 26.9815386)

lmp.pair_style("grace/fs")
lmp.pair_coeff("*", "*", fs_model_path, element)

print("Number of atoms:", lmp.get_natoms())

## 4. Static energy evaluation

Thermodynamic keywords (`pe`, `temp`, `press`, ...) are available as plain attributes and are computed on demand, independent of what `thermo_style` prints.

In [ ]:
lmp.run(0)
print("Potential energy (eV):", lmp.pe)

## 5. Short NVT molecular-dynamics run

We step the trajectory in chunks of 25 MD steps and read back the instantaneous temperature and total energy after each chunk - the kind of interactive, step-by-step control that the library interface (as opposed to a static input file) makes easy.

In [ ]:
lmp.velocity("all", "create", 500.0, 12345)
lmp.fix("nvt_fix", "all", "nvt", "temp", 500.0, 500.0, 0.1)
lmp.timestep(0.001)

n_chunks = 20
steps_per_chunk = 25

steps, temperatures, energies = [], [], []
for i in range(n_chunks):
    lmp.run(steps_per_chunk)
    steps.append((i + 1) * steps_per_chunk)
    temperatures.append(lmp.temp)
    energies.append(lmp.etotal)

lmp.unfix("nvt_fix")

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(steps, temperatures, marker="o")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Temperature (K)")
axes[0].set_title("NVT thermostat (target: 500 K)")

axes[1].plot(steps, energies, marker="o", color="tab:orange")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Total energy (eV)")
axes[1].set_title("Energy conservation")

fig.tight_layout()
plt.show()

## 6. Shut down the LAMMPS instance

In [ ]:
lmp.close()